# Cervello-Bambino — training su Colab

Notebook per eseguire le fasi di training intensive (PyTorch, `cervello/`) su GPU Colab invece che sul PC locale (solo CPU).

Prima di lanciare le celle: **Runtime -> Cambia tipo di runtime -> GPU**.

## 1. Verifica GPU

In [ ]:
!nvidia-smi

## 2. Clona il repository

In [ ]:
REPO_URL = "https://github.com/Mecks3D/llm-test.git"

# Torna sempre a /content prima di clonare: rieseguire questa cella
# mentre si e' gia' dentro llm-test clonerebbe il repo dentro se stesso.
%cd /content
!rm -rf llm-test
!git clone $REPO_URL
%cd llm-test


## 3. Installa le dipendenze

In [ ]:
!pip install -q -r requirements.txt

## 4. Verifica che torch veda la GPU

In [ ]:
import torch
print("CUDA disponibile:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 5. Monta Google Drive (per salvare i checkpoint)

La sessione Colab perde tutto alla scadenza: i checkpoint vanno salvati su Drive, non nel filesystem locale del runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/cervello-bambino/checkpoint"
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Checkpoint dir:", CHECKPOINT_DIR)

## 6. Smoke test: la suite di test passa nell'ambiente Colab?

Prima di lanciare training lunghi, verifica che l'ambiente sia equivalente a quello locale.

In [ ]:
!python -m pytest -q

## 6b. Canary di apprendimento (Fase 2a, gruppo 6)

Test "cancello duro": 32 esempi di stadio 1, overfit fino a esattezza-risposta 100% (max 1000 step). Marcato `slow` (escluso dalla suite di default): su CPU e' impraticabile, va eseguito qui con GPU.

In [ ]:
!python -m pytest -q -m slow -s


## 6c. Run di fumo (Fase 2a, tappa T6)

Prima del training vero: genera il dataset ridotto (200 storie, solo stadio 1) e allena per 300 step con `configs/v1_fumo.yaml`. Verifica che la loss scenda e che compaiano log/checkpoint in `dati/risultati/v1_fumo/`. Le prime valutazioni possono essere lente: il modello appena inizializzato non ha ancora imparato a emettere `[FINE]`, quindi la decodifica greedy può arrivare fino al tetto `ctx=3072` prima di fermarsi. Se una singola valutazione impiega più di qualche minuto, interrompi e segnalalo.

In [ ]:
!python -m esami.genera --config configs/v1_fumo.yaml --stadio 1


In [ ]:
!python -m cervello.addestra --config configs/v1_fumo.yaml --stadio 1


In [ ]:
import json, shutil, os

with open('dati/risultati/v1_fumo/log.jsonl', encoding='utf-8') as f:
    righe = [json.loads(r) for r in f]
for r in righe:
    print(r)

# Esito dell'esame (esattezza per tipo + conteggi di calibrazione:
# 'errore' = struttura giusta ma contenuto sbagliato -> poco training;
# 'malformata' dominante -> problema di decodifica, indagare).
with open('dati/risultati/v1_fumo/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# Il filesystem di Colab e' effimero: copia i risultati su Drive.
dest = os.path.join(CHECKPOINT_DIR, 'v1_fumo')
shutil.copytree('dati/risultati/v1_fumo', dest, dirs_exist_ok=True)
print('risultati copiati su Drive ->', dest)


## 6d. Run di fumo LUNGO (verifica post-revisione T1-T6)

Stadio 1 con 1000 storie e max 3000 step (`configs/v1_fumo_lungo.yaml`, ~20-30 min).
Obiettivo: vedere `esattezza_dev` **salire sopra lo zero** (conferma che la catena
training→decodifica→grafo impara anche fuori dall'overfit del canary) e misurare
quanti step servono per avvicinare la soglia 0.95 — l'ancora per stimare la durata
del curriculum completo (decisione 10). Se raggiunge soglia+0.01 per 2 valutazioni
consecutive si ferma da solo prima dei 3000 step.


In [ ]:
!python -m esami.genera --config configs/v1_fumo_lungo.yaml --stadio 1


In [ ]:
!python -m cervello.addestra --config configs/v1_fumo_lungo.yaml --stadio 1


In [ ]:
import json, shutil, os

with open('dati/risultati/v1_fumo_lungo/log.jsonl', encoding='utf-8') as f:
    righe = [json.loads(r) for r in f]
for r in righe:
    print(r)

# Esito dell'esame, con i primi campioni non esatti (generato vs oro):
# le 'malformate' si guardano, non si indovinano.
with open('dati/risultati/v1_fumo_lungo/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# Il filesystem di Colab e' effimero: copia i risultati su Drive.
dest = os.path.join(CHECKPOINT_DIR, 'v1_fumo_lungo')
shutil.copytree('dati/risultati/v1_fumo_lungo', dest, dirs_exist_ok=True)
print('risultati copiati su Drive ->', dest)


## 6e. Prova del checkpoint intra-stadio su Drive (da fare PRIMA di T7)

Prova end-to-end, tutta automatica (~5 minuti): lancia il run di fumo con
`--copia-sicurezza`, lo **uccide** (SIGKILL, come un runtime che muore) appena il
primo checkpoint parziale compare su Drive, cancella la cartella locale dei
risultati (come farebbe un runtime nuovo), poi rilancia lo stesso comando e
verifica che riprenda da Drive e arrivi in fondo. Eseguire le 5 celle in ordine;
l'ultima stampa `PROVA SUPERATA` se tutto funziona.


In [ ]:
# 6e-1: cartella di prova su Drive (pulita) e niente residui locali.
import os, shutil
COPIA_DIR_FUMO = os.path.join(CHECKPOINT_DIR, 'v1_fumo_prova')
shutil.rmtree(COPIA_DIR_FUMO, ignore_errors=True)
os.makedirs(COPIA_DIR_FUMO, exist_ok=True)
shutil.rmtree('dati/risultati/v1_fumo', ignore_errors=True)
print('cartella di prova su Drive:', COPIA_DIR_FUMO)


In [ ]:
# 6e-2: dataset di fumo (idempotente, veloce).
!python -m esami.genera --config configs/v1_fumo.yaml --stadio 1


In [ ]:
# 6e-3: run interrotto. Il training parte in un sottoprocesso e viene UCCISO
# appena stadio1_parziale.pt compare su Drive (prima valutazione, step 100,
# ~1-2 min). Poi si cancella la cartella locale: resta SOLO la copia su Drive.
import os, shutil, subprocess, time

# Si aspettano ENTRAMBI i file (parziale + log): vengono copiati in
# quest'ordine e uccidere in mezzo lascerebbe il log indietro di un giro.
parziale_drive = os.path.join(COPIA_DIR_FUMO, 'stadio1_parziale.pt')
log_drive = os.path.join(COPIA_DIR_FUMO, 'log.jsonl')
with open('fumo_interrotto.log', 'w') as log:
    proc = subprocess.Popen(
        ['python', '-m', 'cervello.addestra', '--config', 'configs/v1_fumo.yaml',
         '--stadio', '1', '--copia-sicurezza', COPIA_DIR_FUMO],
        stdout=log, stderr=subprocess.STDOUT)
    attesa = 0
    while proc.poll() is None and not (os.path.exists(parziale_drive) and os.path.exists(log_drive)):
        time.sleep(5)
        attesa += 5
        if attesa % 30 == 0:
            print(f'... in attesa del primo parziale su Drive ({attesa}s)')
    if proc.poll() is not None:
        print(open('fumo_interrotto.log').read())
        raise RuntimeError('il training e\' finito senza mai scrivere il parziale su Drive')
    proc.kill()
    proc.wait()

print(open('fumo_interrotto.log').read())
print('>>> training UCCISO dopo il primo parziale su Drive:', parziale_drive)
shutil.rmtree('dati/risultati/v1_fumo', ignore_errors=True)
print('>>> cartella locale dei risultati cancellata (simula un runtime nuovo)')


In [ ]:
# 6e-4: ripresa. Stesso identico comando: deve stampare 'recuperato dalla
# copia di sicurezza' e 'ripresa intra-stadio da ... step 100', poi finire
# i 300 step e l'esame (che a 300 step FALLISCE: atteso, non e' la prova).
!python -m cervello.addestra --config configs/v1_fumo.yaml --stadio 1 --copia-sicurezza "{COPIA_DIR_FUMO}"


In [ ]:
# 6e-5: verifica finale.
import os, json

for nome in ('stadio1.pt', 'esame_stadio1.json', 'log.jsonl'):
    p = os.path.join(COPIA_DIR_FUMO, nome)
    assert os.path.exists(p) and os.path.getsize(p) > 0, f'manca su Drive: {p}'
assert not os.path.exists(os.path.join(COPIA_DIR_FUMO, 'stadio1_parziale.pt')), \
    'il parziale doveva sparire a stadio completato'

with open('dati/risultati/v1_fumo/log.jsonl', encoding='utf-8') as f:
    steps = [json.loads(r)['step'] for r in f]
print('step nel log (curva intera, senza buchi):', steps)
assert steps == [100, 200, 300], steps

print()
print('PROVA SUPERATA: parziale su Drive, ripresa automatica, log intero,')
print('pulizia a fine stadio. Si puo\' procedere con la sezione 7 (T7).')


## 7. T7 — Stadio 1 di produzione (config v1)

Primo stadio del run vero: 5000 storie corte, max 20.000 step, eval ogni 500
(`configs/v1.yaml`). **Stima: ~2,5-3h nel caso peggiore** (meno se l'esattezza dev
raggiunge soglia+0.01 per 2 valutazioni consecutive e si ferma da solo) — run > 2h,
va lanciato solo con decisione esplicita (FASE2_PIANO.md, decisione 10).
Al termine esegue l'esame di stadio 1 (soglia 0.95) e salva checkpoint + esito.
Gli stadi 2-3 si lanciano DOPO, con `--stadio 2` / `--stadio 3` (riprendono dal
checkpoint precedente), una volta ristimata la durata dai dati di questo run.

**Checkpoint intra-stadio e ripresa**: a ogni valutazione (500 step) il training
salva `stadio1_parziale.pt` in locale e, grazie a `--copia-sicurezza`, lo replica
subito su Drive (scrittura locale + copia: scrivere direttamente sul mount di
Drive dentro il training loop è notoriamente inaffidabile). Se il runtime muore:
nuovo runtime, celle 1-5 e clone, poi rilanciare le stesse celle qui sotto —
il training recupera da solo il parziale da Drive e riprende (messaggio
`ripresa intra-stadio da ...`). Perdita massima: 500 step (~10-15 min).
Per ripartire invece DA ZERO, cancellare prima `stadio1_parziale.pt` dalla
cartella su Drive.


In [ ]:
# Checkpoint, log ed esiti vengono replicati su Drive dal training stesso
# (--copia-sicurezza, cella successiva): qui si controlla solo se su Drive
# c'e' gia' un parziale da cui il run riprenderebbe.
import os
COPIA_DIR = os.path.join(CHECKPOINT_DIR, 'v1')
os.makedirs(COPIA_DIR, exist_ok=True)
print('copia di sicurezza ->', COPIA_DIR)
parziale = os.path.join(COPIA_DIR, 'stadio1_parziale.pt')
if os.path.exists(parziale):
    print('ATTENZIONE: trovato', parziale)
    print('il training RIPRENDERA\' da li\'; cancellarlo per ripartire da zero.')


In [ ]:
!python -m esami.genera --config configs/v1.yaml --stadio 1


In [ ]:
!python -m cervello.addestra --config configs/v1.yaml --stadio 1 --copia-sicurezza "{COPIA_DIR}"


In [ ]:
import json

with open('dati/risultati/v1/log.jsonl', encoding='utf-8') as f:
    righe = [json.loads(r) for r in f]
for r in righe:
    print(r)

with open('dati/risultati/v1/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# Niente copia manuale: checkpoint, log ed esame sono gia' stati replicati
# su Drive dal training (--copia-sicurezza).


## 8. Curriculum completo in un comando solo (riferimento)

`python -m cervello.addestra --config configs/v1.yaml` esegue stadi 1-3 con esami
ai cancelli (il criterio di accettazione "run riproducibile da un comando solo").
Stima nel caso peggiore ~20h: su Colab conviene il percorso per-stadio della
sezione 7. Prima di lanciare gli stadi 2-3 servono anche i loro dataset:
`!python -m esami.genera --config configs/v1.yaml --stadio 2` (e `--stadio 3`).


In [ ]:
!python -m cervello.addestra --config configs/v1.yaml


## 9. Stadio "facile" (cast ridotto) — riparte dal checkpoint di v1

Curriculum separato (`configs/v1_facile.yaml`, run `v1_facile`): storie con
solo 3 personaggi (anna, piero, maria) invece di 6, per ridurre la
confusione di binding entità→luogo osservata nell'esame dello stadio 1 di
v1 (0.573 di esattezza, soglia 0.95 — vedi analisi del log). Non riparte da
pesi casuali: usa `--pesi-iniziali` per caricare i pesi già addestrati di
`v1/stadio1.pt` (già su Drive, replicato dalla sezione 7).

Va lanciato DOPO la sezione 7 (serve `v1/stadio1.pt` su Drive). Dataset e
step molto più leggeri del run di produzione (1500 storie, max 6000 step).

**Stessa protezione da interruzioni della sezione 7**: `--copia-sicurezza`
replica su Drive checkpoint intra-stadio, log ed esito a ogni valutazione;
se il runtime muore, rilanciare le stesse celle qui sotto e il training
riprende da solo dal parziale (messaggio `ripresa intra-stadio da ...`).
Per ripartire da zero, cancellare prima `stadio1_parziale.pt` dalla
cartella `v1_facile` su Drive.

In [ ]:
# Stessa logica della sezione 7: cartella di copia di sicurezza dedicata a
# questa run, piu' il checkpoint di partenza (stadio 1 di v1, cast pieno)
# gia' su Drive.
import os
COPIA_DIR_FACILE = os.path.join(CHECKPOINT_DIR, 'v1_facile')
os.makedirs(COPIA_DIR_FACILE, exist_ok=True)
print('copia di sicurezza ->', COPIA_DIR_FACILE)

PESI_INIZIALI = os.path.join(CHECKPOINT_DIR, 'v1', 'stadio1.pt')
assert os.path.exists(PESI_INIZIALI), (
    f"manca il checkpoint di partenza: {PESI_INIZIALI} "
    "(eseguire prima la sezione 7)"
)
print('pesi iniziali <-', PESI_INIZIALI)

parziale = os.path.join(COPIA_DIR_FACILE, 'stadio1_parziale.pt')
if os.path.exists(parziale):
    print('ATTENZIONE: trovato', parziale)
    print('il training RIPRENDERA\' da li\'; cancellarlo per ripartire da zero.')

In [ ]:
!python -m esami.genera --config configs/v1_facile.yaml --stadio 1

In [ ]:
!python -m cervello.addestra --config configs/v1_facile.yaml --stadio 1 --pesi-iniziali "{PESI_INIZIALI}" --copia-sicurezza "{COPIA_DIR_FACILE}"

In [ ]:
import json

with open('dati/risultati/v1_facile/log.jsonl', encoding='utf-8') as f:
    righe = [json.loads(r) for r in f]
for r in righe:
    print(r)

with open('dati/risultati/v1_facile/esame_stadio1.json', encoding='utf-8') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

# Niente copia manuale: checkpoint, log ed esame sono gia' stati replicati
# su Drive dal training (--copia-sicurezza).